In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

from notebooks.utils.spark_manager import create_spark, read_postgres
from notebooks.utils.pg_connector import postgres

def query(sql, params=None):
    with postgres() as pg:
        return pg.query(sql, params)


SyntaxError: unexpected character after line continuation character (spark_manager.py, line 5)

In [3]:
spark = SparkSession.builder \
    .master("spark://192.168.1.13:7077") \
    .appName("chembl_eda") \
    .config("spark.driver.host", "192.168.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.memory", "6g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.memoryOverhead", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.eventLog.gcMetrics.oldGenerationGarbageCollectors", "G1 Old Generation") \
    .config("spark.eventLog.gcMetrics.youngGenerationGarbageCollectors", "G1 Young Generation") \
    .config("spark.eventLog.dir", "file:////home/patryk-kuszneruk/repos/wszi_2025_2026/tmp/spark-events") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://192.168.1.13:5433/chembl_36"
properties = {
    "user": "chembl",
    "password": "chembl",
    "driver": "org.postgresql.Driver"
}


your 131072x1 screen size is bogus. expect trouble
26/02/02 19:09:08 WARN Utils: Your hostname, DESKTOP-EKMA9QO resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/02 19:09:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/pkuszn/.ivy2/cache
The jars for the packages stored in: /home/pkuszn/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d45ad5f8-bc81-4e14-9b2d-ac7360e2da82;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.6.0!postgresql.jar (107ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.31.0/checker-qual-3.31.0.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.31.0!checker-qual.jar (53ms)
:: resolution report :: resolve 703ms :: artifacts dl 163ms
	:: modules in use:
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	-------------------------------

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from pyspark.sql.functions import broadcast
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct
query = """
(
SELECT 
    activity_id,
    assay_id,
    molregno,
    standard_type,
    CAST(standard_value AS NUMERIC(38,10)) AS standard_value,
    standard_units,
    relationship,
    pchembl_value,
    uid,
    tid,
    data_validity_comment,
    activity_comment,
    published_date,
    document_chembl_id,
    src_id
FROM public.activities
) AS activities_subset
"""

df_activities = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    column="activity_id",
    lowerBound=1,
    upperBound=20873933,
    numPartitions=20,
    properties=properties
)
df_structures = spark.read.jdbc(
    url=jdbc_url,
    table="public.compound_structures",
    column="molregno",
    lowerBound=1,
    upperBound=2854815,
    numPartitions=10,
    properties=properties
)

df_activities = df_activities.withColumn(
    "standard_value",
    col("standard_value").cast("double")
)

df_join = df_activities.join(
    df_structures,
    on="molregno"
).filter("standard_value IS NOT NULL")

df_join.limit(5).show()

df_join.selectExpr("count(*) as total_records").show()

df_join.select(countDistinct("canonical_smiles")).show()

output_path = "/home/patryk-kuszneruk/repos/wszi_2025_2026/parquets/activities.parquet"
df_join.coalesce(20).write.mode("overwrite").parquet(output_path)

Py4JJavaError: An error occurred while calling o71.jdbc.
: org.postgresql.util.PSQLException: ERROR: column "other_columns" does not exist
  Pozycja: 128
	at org.postgresql.core.v3.QueryExecutorImpl.receiveErrorResponse(QueryExecutorImpl.java:2713)
	at org.postgresql.core.v3.QueryExecutorImpl.processResults(QueryExecutorImpl.java:2401)
	at org.postgresql.core.v3.QueryExecutorImpl.execute(QueryExecutorImpl.java:368)
	at org.postgresql.jdbc.PgStatement.executeInternal(PgStatement.java:498)
	at org.postgresql.jdbc.PgStatement.execute(PgStatement.java:415)
	at org.postgresql.jdbc.PgPreparedStatement.executeWithFlags(PgPreparedStatement.java:190)
	at org.postgresql.jdbc.PgPreparedStatement.executeQuery(PgPreparedStatement.java:134)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.getQueryOutputSchema(JDBCRDD.scala:68)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.resolveTable(JDBCRDD.scala:58)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRelation$.getSc

# Liczba unikalnych molekuł

In [ ]:
n_rows = len(df_join)
n_smiles = df_join['canonical_smiles'].nunique()
print(f"Liczba rekordów: {n_rows}")
print(f"Liczba unikalnych SMILES: {n_smiles}")
print(f"Średnia liczba assayów na molekułę: {n_rows / n_smiles:.2f}")

# Duplikaty SMILES

In [ ]:
dups = df_join.duplicated(subset=['canonical_smiles','standard_value'], keep=False)
print(f"Liczba duplikatów: {dups.sum()}")
df_join[dups].head()

# Filtracja IC50 / nM

In [ ]:
df_filtered = df_join[
    (df_join['standard_type'] == 'IC50') &
    (df_join['standard_units'] == 'nM')
].copy()

# Transformacja IC50 -> pIC50

In [ ]:
import numpy as np
df_filtered['pIC50'] = -np.log10(df_filtered['standard_value'] * 1e-9)
df_filtered[['standard_value','pIC50']].head()

In [ ]:
plt.hist(df_filtered['pIC50'], bins=50)
plt.xlabel("pIC50")
plt.ylabel("Liczba aktywności")
plt.show()

assays_per_smiles = df_filtered.groupby('canonical_smiles').size()
plt.boxplot(assays_per_smiles)
plt.ylabel("Liczba assayów / molekuła")
plt.show()